# Liputan6 AMR Parsing

## Add Input (all four)
1. **amr-code-modules** - `common/` and `model_interface/` folders
2. **amr-model** - the `mbart-en-id-smaller-concat-finetuned` weights
3. **liputan6-data** - `analysis_data.csv`
4. **liputan6-translate** - the `translate/` folder, saved as a dataset from
   `translate_liputan6.ipynb`'s output

Every stable path is written explicitly in the setup cell. Change each path
directly if the Kaggle username or dataset mount layout differs.

Enable **GPU accelerator (T4)**.

## IMPORTANT - this notebook must be run INTERACTIVELY
Cell 1 builds a Python 3.10 conda env (the parser code needs 3.10, Kaggle's
default kernel is newer), and **you must switch the kernel by hand** afterwards.
A headless *Save & Run All* cannot switch kernels, so it will not work here.

So, unlike the translation notebook:
1. Run **Cell 1**, wait for it to finish.
2. **Kernel -> Change Kernel -> 'Python (3.10) AMR'**.
3. Run the remaining cells.
4. **Click Save Version yourself** when it finishes - nothing is saved automatically.

Turn on **Persistence: Files only** so an idle timeout doesn't wipe progress.

## Chaining runs
- **First run:** set `PREV_DIR = ""`. After saving, make a dataset named
  **`liputan6-amr-graphs`** from the output.
- **Later runs:** leave `PREV_DIR` as-is; already-parsed IDs are skipped.

# 1. Create Virtual Environment with Custom Python Version

In [ ]:
%%bash
PYTHON_VERSION="3.10"
ENV_NAME="amr_env"
ENV_PATH="/kaggle/working/amr_env"
echo "=== Setting up conda environment with Python $PYTHON_VERSION ==="
eval "$(conda shell.bash hook)"
if [ ! -d "$ENV_PATH" ]; then
    conda create -y -p "$ENV_PATH" python=$PYTHON_VERSION
else
    echo "Environment already exists at $ENV_PATH"
fi
conda activate "$ENV_PATH"
pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
pip install --quiet transformers penman>=1.1.0 sentencepiece sacremoses regex networkx pandas tqdm ipykernel
python -m ipykernel install --user --name=$ENV_NAME --display-name "Python ($PYTHON_VERSION) AMR"
echo "=== Setup Complete ==="
echo ">>> Kernel > Change Kernel > 'Python (3.10) AMR', then run from Cell 2."

## After Cell 1 completes:
1. **Kernel -> Change Kernel**
2. Select **'Python (3.10) AMR'**
3. Continue from Cell 2 below

# 2. Setup, Imports & Paths

In [ ]:
import os, sys, json, shutil, torch, penman, numpy as np, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================================
# KAGGLE PATHS - all stable paths are explicit and independent.
# ============================================================
CODE_MODULES_PATH = "/kaggle/input/datasets/fedrianzdharma/amr-code-modules"
MODEL_PATH        = "/kaggle/input/datasets/fedrianzdharma/amr-model"
ANALYSIS_CSV_PATH = "/kaggle/input/datasets/fedrianzdharma/liputan6-data/analysis_data.csv"
TRANSLATE_DIR     = "/kaggle/input/datasets/fedrianzdharma/liputan6-translate/translate"
OUTPUT_DIR        = "/kaggle/working/amr_graphs"
ZIP_PATH          = "/kaggle/working/amr_graphs_liputan6.zip"

# CROSS-SESSION RESUME: use an empty string on the first run.
PREV_DIR          = "/kaggle/input/datasets/fedrianzdharma/liputan6-amr-graphs/amr_graphs"
# PREV_DIR = ""   # <- use this on the very first run

# ============================================================
# RUN BUDGET - this notebook runs INTERACTIVELY (kernel switch), so an idle
# timeout can end the session. Keep each run bounded, then Save Version.
#   MAX_NEW      : stop after this many NEW graphs this run (None = no cap)
#   TIME_BUDGET_H: also stop after this many hours, whichever comes first
# ============================================================
MAX_NEW       = 40000
TIME_BUDGET_H = 10.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

if CODE_MODULES_PATH not in sys.path:
    sys.path.insert(0, CODE_MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus

print("Python  :", sys.version.split()[0])
print("Torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available())
for name, p in [("code modules", CODE_MODULES_PATH), ("model", MODEL_PATH),
                ("analysis csv", ANALYSIS_CSV_PATH), ("translate", TRANSLATE_DIR),
                ("prev output", PREV_DIR)]:
    print(f"{name:13s} exists: {os.path.exists(p) if p else False}  ({p})")

# 3. Load Model & Tokenizer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Default Device :", device)

amr_config = AutoConfig.from_pretrained(MODEL_PATH)
amr_tokenizer = AMRBartTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
amr_model = MBartForConditionalGeneration.from_pretrained(MODEL_PATH, config=amr_config)
amr_model.resize_token_embeddings(len(amr_tokenizer))
amr_model = amr_model.to(device)
amr_model.eval()
print("Model loaded on :", amr_model.device)

# 4. Dataset & DataLoader

In [ ]:
class Liputan6Dataset(Dataset):
    # Only rows that (1) have a translation file and (2) are not yet parsed.
    def __init__(self, analysis_csv_path=ANALYSIS_CSV_PATH, translate_dir=TRANSLATE_DIR, output_dir=OUTPUT_DIR):
        df = pd.read_csv(analysis_csv_path, dtype={"id": str})

        available = set()
        if os.path.isdir(translate_dir):
            for fn in os.listdir(translate_dir):
                if fn.endswith(".txt"):
                    available.add(fn[:-4])

        # already-parsed = this run's output PLUS the previous run's mounted
        # output, so the job resumes across sessions
        parsed = set()
        for d in [PREV_DIR, output_dir]:
            if d and os.path.isdir(d):
                for fn in os.listdir(d):
                    if fn.endswith(".txt"):
                        parsed.add(fn[:-4])

        total = len(df)
        mask_tr = df["id"].isin(available)
        mask_np = ~df["id"].isin(parsed)
        self.df = df[mask_tr & mask_np].reset_index(drop=True)
        self.translate_dir = translate_dir

        print(f"Total rows in CSV        : {total}")
        print(f"Translations available   : {int(mask_tr.sum())}")
        print(f"Already parsed (skipped) : {int((mask_tr & ~mask_np).sum())}")
        print(f"Remaining to parse       : {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        with open(os.path.join(self.translate_dir, row["id"] + ".txt"), "r", encoding="utf-8") as f:
            translated = f.read()
        return {"id": row["id"], "text": row["text"], "translated_text": translated}

In [ ]:
def add_amr_mask(tokenized_inputs, masks):
    amr_suffix = [amr_tokenizer.amr_bos_token_id, amr_tokenizer.mask_token_id, amr_tokenizer.amr_eos_token_id]
    amr_mask_suffix = [1, 1, 1]
    updated_inputs, updated_masks = [], []
    for input_ids, mask in zip(tokenized_inputs, masks):
        pad_start = mask.index(0) if 0 in mask else len(mask)
        updated_inputs.append(input_ids[:pad_start] + amr_suffix + input_ids[pad_start:])
        updated_masks.append(mask[:pad_start] + amr_mask_suffix + mask[pad_start:])
    return updated_inputs, updated_masks

def wrapper_collate_fn(prefix_lang1, prefix_lang2):
    def collate(batch):
        all_ids, all_texts, all_tr = [], [], []
        for i in batch:
            all_ids.append(i["id"]); all_texts.append(i["text"]); all_tr.append(i["translated_text"])
        all_inputs = [f"{prefix_lang1} {t} {prefix_lang2} {tr}" for t, tr in zip(all_texts, all_tr)]
        tok = amr_tokenizer(all_inputs, max_length=None, padding=True, truncation=True)
        return (all_ids,) + tuple(add_amr_mask(tok["input_ids"], tok["attention_mask"]))
    return collate

ds = Liputan6Dataset()
loader = DataLoader(ds, batch_size=1, collate_fn=wrapper_collate_fn("id_ID", "en_XX"))

# 5. Parsing Functions

In [ ]:
def decode_amr_output(pred_token_ids, tokenizer):
    pred_ids = list(pred_token_ids)
    pred_ids[0] = tokenizer.bos_token_id
    pred_ids = [tokenizer.eos_token_id if tok == tokenizer.amr_eos_token_id else tok
                for tok in pred_ids if tok != tokenizer.pad_token_id]
    graph, status, _ = tokenizer.decode_amr(pred_ids, restore_name_ops=False)
    return penman.encode(graph), status

def store_graph(ids, amr_strings, output_dir=OUTPUT_DIR):
    os.makedirs(output_dir, exist_ok=True)
    for data_id, amr_str in zip(ids, amr_strings):
        with open(os.path.join(output_dir, f"{data_id}.txt"), "w", encoding="utf-8") as f:
            f.write(amr_str)

# 6. Run AMR Parsing Loop

In [ ]:
import time
start_t = time.time()
budget_s = TIME_BUDGET_H * 3600
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed, stop_reason = 0, "completed all remaining"
print(f"Starting AMR parsing for {len(ds)} samples...")

pbar = tqdm(loader, desc="Parsing AMR")
for batch_ids, inputs, masks in pbar:
    # --- stop cleanly so you can Save Version with a consistent output ---
    if MAX_NEW is not None and total_parsed >= MAX_NEW:
        stop_reason = f"hit MAX_NEW={MAX_NEW}"
        break
    if time.time() - start_t > budget_s:
        stop_reason = f"hit TIME_BUDGET_H={TIME_BUDGET_H}"
        break

    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=5, max_length=1024,
                decoder_start_token_id=amr_tokenizer.amr_bos_token_id)
        amr_strings = []
        for i in range(outputs.shape[0]):
            amr_string, status = decode_amr_output(outputs[i].cpu().tolist(), amr_tokenizer)
            amr_strings.append(amr_string)
            name = status.name if hasattr(status, "name") else str(status)
            status_counts[name if name in status_counts else "ERROR"] += 1
        store_graph(batch_ids, amr_strings)
        total_parsed += len(batch_ids)
    except Exception:
        status_counts["ERROR"] += len(batch_ids)
        continue
    if total_parsed % 500 == 0:
        torch.cuda.empty_cache()
    pbar.set_postfix(new=total_parsed, elapsed_h=f"{(time.time()-start_t)/3600:.2f}")

pbar.close()
print("")
print(f"Stopped because : {stop_reason}")
print(f"Parsed this run : {total_parsed}")
print(f"Elapsed         : {(time.time()-start_t)/3600:.2f} h")
for k, v in status_counts.items():
    if v:
        print(f"  {k:8s}: {v} ({v/max(total_parsed,1)*100:.1f}%)")

# 7. Zip Output for Download

In [ ]:
# Keep this version cumulative: copy forward graphs from the previous run.
copied = 0
if PREV_DIR and os.path.isdir(PREV_DIR):
    have = set(os.listdir(OUTPUT_DIR))
    for fn in tqdm(os.listdir(PREV_DIR), desc="Carrying forward"):
        if fn.endswith(".txt") and fn not in have:
            shutil.copyfile(os.path.join(PREV_DIR, fn), os.path.join(OUTPUT_DIR, fn))
            copied += 1
total = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.txt')])
print(f"Carried forward: {copied} | TOTAL graphs in output: {total}")
print("\n>>> Click 'Save Version' or this output is lost at session end.")

In [ ]:
import zipfile
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")]
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in tqdm(files, desc="Zipping"):
        zf.write(os.path.join(OUTPUT_DIR, fn), fn)
print(f"Done: {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1024/1024:.1f} MB), {len(files)} graphs")
print("Download it from the Output tab on the right.")